# Gemma 3 12B Multilingual WMT Translation Fine-Tuning (INT8)

This notebook handles the joint multilingual fine-tuning of `google/gemma-3-12b-it` on the longest 50,000 sentences across Czech-German, English-Chinese, and English-Arabic translation pairs.

**Core Features:**
1. **INT8 Precision**: Performs fine-tuning in 8-bit to maintain high model quality matching our baseline.
2. **Full Linear Targets**: Adapts all linear blocks (`q_proj`, `k_proj`, `v_proj`, `o_proj`, `gate_proj`, `up_proj`, `down_proj`) for translation.
3. **Custom COMET Checkpointing**: Evaluates system translation quality on FLORES-200 after each epoch using the COMET metric, and automatically saves the absolute best performing checkpoint.

In [ ]:
# Install training and evaluation dependencies
!pip install -q transformers>=4.48.0 datasets peft trl bitsandbytes accelerate huggingface_hub evaluate unbabel-comet pytorch-lightning

In [ ]:
import os

# Redirect Hugging Face Cache to /kaggle/tmp or /tmp to avoid home directory disk space limits
if os.name != "nt":
    if os.path.exists("/kaggle"):
        os.environ["HF_HOME"] = "/kaggle/tmp/huggingface_cache"
        os.environ["HF_DATASETS_CACHE"] = "/kaggle/tmp/huggingface_cache/datasets"
        os.environ["HF_HUB_CACHE"] = "/kaggle/tmp/huggingface_cache/hub"
        try:
            get_ipython().run_line_magic('env', 'HF_HOME=/kaggle/tmp/huggingface_cache')
            get_ipython().run_line_magic('env', 'HF_DATASETS_CACHE=/kaggle/tmp/huggingface_cache/datasets')
            get_ipython().run_line_magic('env', 'HF_HUB_CACHE=/kaggle/tmp/huggingface_cache/hub')
        except Exception:
            pass
    else:
        os.environ["HF_HOME"] = "/tmp/huggingface_cache"
        os.environ["HF_DATASETS_CACHE"] = "/tmp/huggingface_cache/datasets"
        os.environ["HF_HUB_CACHE"] = "/tmp/huggingface_cache/hub"
        try:
            get_ipython().run_line_magic('env', 'HF_HOME=/tmp/huggingface_cache')
            get_ipython().run_line_magic('env', 'HF_DATASETS_CACHE=/tmp/huggingface_cache/datasets')
            get_ipython().run_line_magic('env', 'HF_HUB_CACHE=/tmp/huggingface_cache/hub')
        except Exception:
            pass

# Create outputs directory
output_dir = "/kaggle/working/gemma3-12b-wmt-lora"
os.makedirs(output_dir, exist_ok=True)
print(f"✅ Save directory ready: {output_dir}")

In [ ]:
import os
from huggingface_hub import login

# Check if HF_TOKEN is in environment or Kaggle Secrets
hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient
        hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass

if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        pass

if hf_token and hf_token.strip():
    print("🔑 Hugging Face token found in secrets/environment. Logging in...")
    login(token=hf_token.strip())
    os.environ["HF_TOKEN"] = hf_token.strip()
    try:
        get_ipython().run_line_magic('env', f'HF_TOKEN={hf_token.strip()}')
    except Exception:
        pass
else:
    hf_token = input("Enter your Hugging Face WRITE Token (optional, press Enter to skip): ")
    if hf_token.strip():
        login(token=hf_token.strip())
        os.environ["HF_TOKEN"] = hf_token.strip()
        try:
            get_ipython().run_line_magic('env', f'HF_TOKEN={hf_token.strip()}')
        except Exception:
            pass

## Run Joint Multilingual Fine-Tuning Pipeline

We run the structured python script `src/finetune.py`. This script handles the loading, length-sorting, 8-bit model quantization, custom training loop, and COMET evaluation callbacks.

**Parameters:**
- `--model_id`: Model name (`google/gemma-3-12b-it`)
- `--epochs`: Number of training epochs (e.g. 50)
- `--subset_size`: Dataset subset size (e.g. 50000 longest sentences)
- `--output_dir`: Location to save results

In [ ]:
# Trigger training in INT8 precision
!python src/finetune.py \
  --model_id "google/gemma-3-12b-it" \
  --output_dir "/kaggle/working/gemma3-12b-wmt-lora" \
  --epochs 50 \
  --subset_size 50000

## Zip and Download Best Model

Once training completes, the best performing model based on the FLORES-200 COMET evaluations will be located in `/kaggle/working/gemma3-12b-wmt-lora/best_model`.

Run the cell below to zip the best model adapter and generate a download link.

In [ ]:
import shutil
import os
from IPython.display import FileLink

best_model_path = '/kaggle/working/gemma3-12b-wmt-lora/best_model'
zip_name = '/kaggle/working/gemma3-12b-wmt-lora-best-model'

if os.path.exists(best_model_path):
    total_size = sum(
        os.path.getsize(os.path.join(dirpath, f))
        for dirpath, _, filenames in os.walk(best_model_path)
        for f in filenames
    )
    print(f"📦 Best LoRA Adapter Size: {total_size / (1024 * 1024):.2f} MB")
    
    print("Zipping best performing LoRA adapter...")
    shutil.make_archive(zip_name, 'zip', best_model_path)
    
    print("\n✅ Click the link below to download your best performing model adapter!")
    display(FileLink(r'gemma3-12b-wmt-lora-best-model.zip'))
else:
    print("❌ Best model folder not found. Make sure training completed at least one epoch and generated validation metrics.")